In [1]:
# === CELL 0: DIAGNOSTIC CELL ===
import requests
from bs4 import BeautifulSoup

query = "UPI fraud"
page = 1
search_url = f"https://medianama.com/?s={query.replace(' ', '+')}&paged={page}"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://medianama.com"
}

try:
    print(f"Fetching: {search_url}")
    resp = requests.get(search_url, headers=headers, timeout=15)
    print(f"HTTP Status: {resp.status_code}")
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Identify article listing containers
    articles = soup.find_all('article')
    if not articles:
        articles = soup.find_all('div', class_='jeg_post')
    if not articles:
        articles = soup.find_all('div', class_='post-item')
        
    if articles:
        first = articles[0]
        print(f"\nContainer tag/class: <{first.name} class='{first.get('class', [])}'>")
        print("\n--- Raw HTML (first 1500 chars) ---")
        print(first.prettify()[:1500])
        
        # Extract title and URL
        title_heading = first.find('h2', class_='entry-title') or first.find('h2', class_='jeg_post_title') or first.find('h3')
        if title_heading and title_heading.find('a'):
            a_tag = title_heading.find('a')
            print(f"\nExtracted Title: {a_tag.get_text(strip=True)}")
            print(f"Extracted URL: {a_tag.get('href')}")
        else:
            print("\nCould not find title <a> tag in this container.")
    else:
        print("No article containers found.")
except Exception as e:
    print(f"Error: {e}")

Fetching: https://medianama.com/?s=UPI+fraud&paged=1
HTTP Status: 200

Container tag/class: <article class='['post-265028', 'post', 'type-post', 'status-publish', 'format-standard', 'has-post-thumbnail', 'hentry', 'category-news', 'tag-ai', 'tag-digital-payment', 'tag-fintech-lending', 'tag-npci', 'tag-upi']'>

--- Raw HTML (first 1500 chars) ---
<article class="post-265028 post type-post status-publish format-standard has-post-thumbnail hentry category-news tag-ai tag-digital-payment tag-fintech-lending tag-npci tag-upi" id="post-265028">
 <div class="card">
  <a class="card__media" href="https://www.medianama.com/2025/04/223-npci-ai-upi-fraud-detection/" title="NPCI Pilots AI Models to Curb UPI Fraud After Rs 1,087 Crore in Losses">
   <img alt="" class="attachment-post-thumbnail size-post-thumbnail wp-post-image" decoding="async" fetchpriority="high" height="188" sizes="(max-width: 334px) 100vw, 334px" src="https://www.medianama.com/wp-content/uploads/2024/02/ai-8529399_1920-1024x57

In [2]:
# === CELL 1: IMPORTS AND CONFIGURATION ===
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time
import random
import os
import re
from datetime import datetime

# ── ID Continuity ──────────────────────────────────────────────────────────
# The last Unique ID used by indiankanoon_scraper.ipynb is NA-13176.
# This scraper begins from NA-13177.
existing_max_id = 13176  # hardcoded fallback

# Optionally auto-detect from existing master CSVs to be safe
for master_csv in ["data/indiankanoon_complaints.csv", "data/consumer_complaints.csv"]:
    if os.path.exists(master_csv):
        df_existing = pd.read_csv(master_csv)
        if "Unique ID" in df_existing.columns:
            ids = df_existing["Unique ID"].dropna().str.extract(r'(\d+)').astype(float)
            if not ids.empty:
                existing_max_id = max(existing_max_id, int(ids.max().iloc[0]))
                
next_id = existing_max_id + 1
print(f"Starting Unique ID: NA-{next_id:04d}")

# ── Paths ──────────────────────────────────────────────────────────────────
BASE_URL = "https://medianama.com"
SEARCH_URL = "https://medianama.com/"
OUTPUT_CSV = "data/medianama_complaints.csv"
OUTPUT_JSON = "data/medianama.json"
TXT_DIR = "data/txt"

os.makedirs(TXT_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)

MAX_PAGES = 5  # pages per keyword

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Referer": "https://medianama.com"
}

if 'session' not in globals():
    session = requests.Session()

Starting Unique ID: NA-13177


In [3]:
# === CELL 2: KEYWORD TAXONOMY ===
KEYWORD_TAXONOMY = {
    "General Cybercrime / Cyber Fraud Terms": {
        "General Cybercrime": ["cyber crime", "cybercrime", "cyber fraud", "online fraud", "internet fraud", "digital fraud"],
        "Cyber Scam": ["cyber scam", "online scam", "internet scam", "online cheating"],
        "Financial Cyber Fraud": ["financial fraud online", "net banking fraud", "e-banking fraud"]
    },
    "UPI and Digital Payment Fraud": {
        "UPI Fraud": ["UPI fraud", "UPI scam", "Google Pay fraud", "PhonePe fraud", "Paytm fraud", "BHIM fraud"],
        "QR Code Fraud": ["QR code scam", "QR code fraud", "scan and pay fraud"],
        "Payment Link Fraud": ["payment link fraud", "collect request scam", "fake payment link"],
        "Mobile Wallet Fraud": ["mobile wallet fraud", "e-wallet scam", "digital wallet fraud"]
    },
    "OTP and Authentication Fraud": {
        "OTP Fraud": ["OTP fraud", "OTP scam", "OTP theft"],
        "SIM Swap": ["SIM swap fraud", "SIM cloning", "duplicate SIM fraud"],
        "KYC Fraud": ["KYC fraud", "KYC scam", "Aadhaar KYC scam"]
    },
    "Digital Arrest Scam": {
        "Digital Arrest": ["digital arrest", "digital arrest scam", "fake arrest", "video call arrest"],
        "Impersonation Scam": ["police impersonation scam", "CBI fraud call", "customs fraud call", "TRAI scam call", "ED scam call"],
        "Video Call Coercion": ["video call scam", "video call blackmail", "video call extortion", "fake interrogation"]
    },
    "Phishing, Vishing, and Smishing": {
        "Phishing": ["phishing", "phishing attack", "phishing email", "fake website", "spoof website"],
        "Vishing": ["vishing", "voice phishing", "fraud call", "fake bank call"],
        "Smishing": ["smishing", "SMS fraud", "SMS scam", "phishing SMS"]
    },
    "Online Lending and Loan App Fraud": {
        "Loan App Fraud": ["loan app fraud", "instant loan scam", "loan app harassment", "illegal loan app"],
        "Loan App Extortion": ["loan app blackmail", "loan app threat", "morphed photos loan", "recovery agent threat"]
    },
    "Investment and Trading Fraud": {
        "Investment Scam": ["investment scam", "Ponzi scheme", "online investment fraud", "crypto scam", "bitcoin fraud", "forex trading scam", "pig butchering"],
        "Stock Market Fraud": ["stock market scam", "share trading fraud", "demat fraud", "pump and dump"],
        "Task Scam": ["task fraud", "part time job scam", "work from home scam", "Telegram task scam"]
    },
    "Identity Theft and Data Breach": {
        "Identity Theft": ["identity theft", "identity fraud", "Aadhaar misuse", "PAN fraud"],
        "Data Breach": ["data breach", "data leak", "data theft", "customer data breach"]
    },
    "Social Engineering and Romance/Sextortion": {
        "Social Engineering": ["social engineering fraud", "manipulation scam", "trust scam"],
        "Romance Scam": ["romance scam", "dating fraud", "matrimonial fraud", "honey trap", "catfishing fraud"],
        "Sextortion": ["sextortion", "webcam blackmail", "nude video blackmail"]
    },
    "E-Commerce and Delivery Fraud": {
        "E-Commerce Fraud": ["e-commerce fraud", "online shopping fraud", "fake product scam", "Flipkart fraud", "Amazon fraud", "refund scam"],
        "Delivery Fraud": ["fake delivery", "courier fraud", "customs duty scam", "parcel scam", "delivery OTP scam"]
    },
    "Ransomware and Malware": {
        "Ransomware": ["ransomware attack", "cyber ransom", "data encryption attack"],
        "Banking Malware": ["banking trojan", "banking malware", "keylogger fraud", "AnyDesk fraud", "TeamViewer scam", "screen sharing scam"]
    },
    "Emerging and Miscellaneous Fraud Types": {
        "Deepfake Fraud": ["deepfake scam", "deepfake fraud", "AI voice scam", "voice cloning scam"],
        "Utility Scam": ["electricity bill scam", "utility fraud", "disconnection scam"],
        "Aadhaar Fraud": ["Aadhaar fraud", "Aadhaar scam", "biometric fraud", "AEPS fraud"],
        "Cyber Stalking": ["cyber stalking", "cyber bullying", "online harassment", "digital harassment"]
    }
}

In [4]:
# === CELL 3: HELPER FUNCTIONS ===
def classify_narrative_type(text):
    text_lower = text.lower()
    if any(x in text_lower for x in ["i lost", "cheated", "defrauded", "victim", "amount deducted", "money lost"]):
        return "VICTIM"
    elif any(x in text_lower for x in ["attempted", "tried to", "almost", "suspicious", "did not share", "narrowly"]):
        return "NEAR-MISS"
    return "THIRD-PARTY"

def clean_text(text):
    """Remove excessive whitespace and normalize."""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def extract_article_date(soup):
    # Try different date selectors common in WordPress and Medianama themes
    time_entry = soup.find('time', class_='entry-date')
    if time_entry and time_entry.get('datetime'):
        return time_entry.get('datetime').split('T')[0]
        
    time_tag = soup.find('time')
    if time_tag and time_tag.get('datetime'):
        return time_tag.get('datetime').split('T')[0]
        
    jeg_meta = soup.find('span', class_='jeg_meta_date')
    if jeg_meta:
        return clean_text(jeg_meta.get_text(strip=True))
        
    span_date = soup.find('span', class_='date')
    if span_date:
        return clean_text(span_date.get_text(strip=True))
        
    if time_tag:
        return clean_text(time_tag.get_text(strip=True))
        
    return "Unknown Date"

In [5]:
# === CELL 4: SEARCH AND SCRAPING FUNCTIONS ===
def search_medianama(query, page=1):
    results = []
    seen_urls = set()
    
    url = f"https://medianama.com/?s={query.replace(' ', '+')}&paged={page}"
    try:
        time.sleep(random.uniform(2.0, 4.5))
        response = session.get(url, headers=HEADERS, timeout=15)
        
        if response.status_code != 200:
            print(f"  Failed: status {response.status_code}")
            return results
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Selectors in order of preference
        articles = soup.find_all('article')
        if not articles:
            articles = soup.find_all('div', class_='jeg_post')
        if not articles:
            articles = soup.find_all('div', class_='post-item')
        if not articles:
            articles = soup.find_all('h2', class_='entry-title')
            
        if not articles:
            return results
            
        for article in articles:
            title_tag = article.find('h2', class_='entry-title') or \
                        article.find('h2', class_='jeg_post_title') or \
                        (article if article.name == 'h2' and 'entry-title' in article.get('class', []) else article.find('h3'))
            
            if not title_tag:
                continue
                
            a_tag = title_tag.find('a') if title_tag.name != 'a' else title_tag
            if not a_tag:
                a_tag = article.find('a')
                if not a_tag:
                    continue
                    
            title = a_tag.get_text(strip=True)
            href = a_tag.get('href', '')
            
            if not href or href in seen_urls:
                continue
            seen_urls.add(href)
            
            doc_date = extract_article_date(article)
            
            results.append({
                "title": title,
                "url": href,
                "date": doc_date
            })
            
        print(f"  Found {len(results)} results on page {page}.")
        return results
    except Exception as e:
        print(f"  Error searching '{query}' page {page}: {e}")
        return results

def fetch_article_text(url):
    try:
        time.sleep(random.uniform(3.0, 6.0))
        resp = session.get(url, headers=HEADERS, timeout=20)
        
        if resp.status_code == 404:
            return None, "404 Not Found"
        if "captcha" in resp.url.lower() or resp.status_code == 403:
            return None, "BLOCKED BY CAPTCHA OR 403"
        if resp.status_code != 200:
            return None, f"Status code {resp.status_code}"
            
        soup = BeautifulSoup(resp.text, 'html.parser')
        
        container = soup.find('div', class_='entry-content') or \
                    soup.find('div', class_='jeg_main_content') or \
                    soup.find('div', class_='content-inner') or \
                    soup.find('article')
                    
        if not container:
            return None, "No content div found"
            
        # Decompose unwanted elements
        for tag in container.find_all(['script', 'style', 'aside', 'nav', 'footer']):
            tag.decompose()
            
        paragraphs = container.find_all('p')
        if not paragraphs:
            text = container.get_text(separator='\n\n', strip=True)
        else:
            text = "\n\n".join([p.get_text(separator=' ', strip=True) for p in paragraphs if p.get_text(strip=True)])
            
        text = clean_text(text)
        if not text:
            return None, "No text after cleaning"
            
        return text, "Success"
    except requests.exceptions.Timeout:
        return None, "Timeout"
    except Exception as e:
        return None, f"Error: {e}"

In [6]:
# === CELL 5: SAFE SAVE FUNCTION ===
def safe_save(df, csv_path, json_path, json_data):
    """Save to temp file first, then rename — avoids PermissionError if Excel has file open."""
    try:
        temp_csv = csv_path + ".tmp"
        df.to_csv(temp_csv, index=False, encoding="utf-8-sig")
        os.replace(temp_csv, csv_path)
        
        temp_json = json_path + ".tmp"
        with open(temp_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, indent=4, ensure_ascii=False)
        os.replace(temp_json, json_path)
        
        print(f"✅ Checkpoint saved: {len(df)} records")
    except Exception as e:
        print(f"⚠️ Checkpoint save failed (data still in memory): {e}")

In [7]:
# === CELL 6: MAIN SCRAPING LOOP ===
try:
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        master_data_json = json.load(f)
    scraped_urls = set(item.get("URL", "") for item in master_data_json)
    print(f"Loaded {len(scraped_urls)} existing scraped articles from JSON.")
except (FileNotFoundError, json.JSONDecodeError):
    master_data_json = []
    scraped_urls = set()
    print("No existing JSON database found, starting fresh.")

new_results_count = 0

for parent_category, subcategories in KEYWORD_TAXONOMY.items():
    print(f"\n--- Processing Category: {parent_category} ---")
    
    for subcat, keyword_list in subcategories.items():
        for word in keyword_list:
            print(f"\n  Keyword: '{word}'")
            for page_num in range(1, MAX_PAGES + 1):
                search_results = search_medianama(word, page=page_num)
                
                if not search_results:
                    break  # stop paginating if empty page
                    
                for res in search_results:
                    if res['url'] in scraped_urls:
                        print(f"    Skipping already scraped: {res['url'][:60]}")
                        continue
                        
                    print(f"    -> Fetching: {res['title'][:55]}...")
                    article_text, status = fetch_article_text(res['url'])
                    
                    if article_text:
                        existing_max_id += 1
                        assigned_id = f"NA-{existing_max_id:04d}"
                        txt_filename = f"{assigned_id}.txt"
                        
                        # Write .txt file
                        with open(os.path.join(TXT_DIR, txt_filename), "w", encoding="utf-8") as f_txt:
                            f_txt.write(article_text)
                            
                        data_row = {
                            "Unique ID": assigned_id,
                            "Date of Collection": datetime.now().strftime("%Y-%m-%d"),
                            "Collector Name": "Soubhik Sarkar",
                            "Source Platform": "Medianama",
                            "Source Publication": "medianama.com",
                            "Original Date": res['date'],
                            "Title/Headline": res['title'],
                            "URL": res['url'],
                            "Search Query Used": word,
                            "Fraud Category": parent_category,
                            "Fraud Subcategory": subcat,
                            "Narrative Type": classify_narrative_type(article_text),
                            "TXT File Name": txt_filename,
                            "Notes": f"Keyword matched: {word}"
                        }
                        
                        master_data_json.append(data_row)
                        scraped_urls.add(res['url'])
                        new_results_count += 1
                        print(f"       ✅ Saved ID: {assigned_id}. Length: {len(article_text)}")
                        
                        # Intermediate save every 25 records
                        if new_results_count % 25 == 0:
                            df_temp = pd.DataFrame(master_data_json)
                            safe_save(df_temp, OUTPUT_CSV, OUTPUT_JSON, master_data_json)
                            print("==== Intermediate Save ====")
                    else:
                        print(f"       ❌ Failed: {res['url'][:60]} | {status}")
                        
            print(f"  Finished parsing pages for '{word}'.")

print(f"\nTotal new articles scraped this session: {new_results_count}!")

No existing JSON database found, starting fresh.

--- Processing Category: General Cybercrime / Cyber Fraud Terms ---

  Keyword: 'cyber crime'
  Found 15 results on page 1.
    -> Fetching: FIRs Remain Low Even As Reported Cyber Crimes Rise, Par...
       ✅ Saved ID: NA-13177. Length: 9600
    -> Fetching: India’s Cyber Crime Centre Rolls Out e-Zero FIR for Cyb...
       ✅ Saved ID: NA-13178. Length: 6019
    -> Fetching: India’s Cyber Crime Centre Mandates Social Media Platfo...
       ✅ Saved ID: NA-13179. Length: 4770
    -> Fetching: Tamil Nadu Cyber Crime Police Issue Advisory Against Us...
       ✅ Saved ID: NA-13180. Length: 4801
    -> Fetching: Ministry of Home Affairs’ Cyber Crime Wing Launches ‘Pr...
       ✅ Saved ID: NA-13181. Length: 2806
    -> Fetching: 983 SIMs deactivated by Haryana police to curb cyber cr...
       ✅ Saved ID: NA-13182. Length: 2884
    -> Fetching: Indian PM calls for global coordination to tackle cyber...
       ✅ Saved ID: NA-13183. Length: 2177


In [8]:
# === CELL 7: FINAL SAVE ===
if new_results_count > 0:
    df_final = pd.DataFrame(master_data_json)
    safe_save(df_final, OUTPUT_CSV, OUTPUT_JSON, master_data_json)
    print(f"✅ Final save complete. Total records: {len(df_final)}")
    display(df_final.tail(3))
else:
    print("No new data to save.")

✅ Checkpoint saved: 2509 records
✅ Final save complete. Total records: 2509


,Unique ID,Date of Collection,Collector Name,Source Platform,Source Publication,Original Date,Title/Headline,URL,Search Query Used,Fraud Category,Fraud Subcategory,Narrative Type,TXT File Name,Notes
2506,NA-15683,2026-04-01,Soubhik Sarkar,Medianama,medianama.com,2024-09-03,Texas Social Media Law Faces Partial Injunctio...,https://www.medianama.com/2024/09/223-texas-so...,digital harassment,Emerging and Miscellaneous Fraud Types,Cyber Stalking,THIRD-PARTY,NA-15683.txt,Keyword matched: digital harassment
2507,NA-15684,2026-04-01,Soubhik Sarkar,Medianama,medianama.com,2024-07-08,UP Police Book Journalists Under Bharatiya Nya...,https://www.medianama.com/2024/07/223-up-polic...,digital harassment,Emerging and Miscellaneous Fraud Types,Cyber Stalking,NEAR-MISS,NA-15684.txt,Keyword matched: digital harassment
2508,NA-15685,2026-04-01,Soubhik Sarkar,Medianama,medianama.com,2024-06-19,Alliance of Rights Organisations Demands Urgen...,https://www.medianama.com/2024/06/223-napm-let...,digital harassment,Emerging and Miscellaneous Fraud Types,Cyber Stalking,THIRD-PARTY,NA-15685.txt,Keyword matched: digital harassment


In [9]:
# === CELL 8: STANDALONE DIAGNOSTIC TEST CELL ===
test_query = "UPI fraud"
print(f"Testing search for: '{test_query}'")

res = search_medianama(test_query, page=1)
if res:
    import json as _json
    print(_json.dumps(res[0], indent=2))
    
    print("\nAttempting article text fetch...")
    txt, stat = fetch_article_text(res[0]['url'])
    
    if txt:
        print(f"Status: {stat}")
        print("-" * 40)
        print(txt[:600] + "...\n[TRUNCATED DIAGNOSTIC VIEW]")
    else:
        print(f"Failed: {stat}")
else:
    print("Search returned no results.")

Testing search for: 'UPI fraud'
  Found 15 results on page 1.
{
  "title": "NPCI Pilots AI Models to Curb UPI Fraud After Rs 1,087 Crore in Losses",
  "url": "https://www.medianama.com/2025/04/223-npci-ai-upi-fraud-detection/",
  "date": "2025-04-03"
}

Attempting article text fetch...
Status: Success
----------------------------------------
The National Payments Corporation of India (NPCI) is testing AI models to curb fraudulent digital transactions, NPCI Chief Risk Officer Viswanath Krishnamurthy disclosed in a recent interaction with the press. Krishnamurthy’s comments come as UPI fraud rose 85% in FY 2023-24 from the previous year, amounting to a loss of Rs 1,087 crore.

The organization is utilizing analytical AI models to identify fraudulent accounts and assign them risk scores. Krishnamurthy cited an example of NPCI’s AI model allocating risk scores to accounts aiming to facilitate multiple transfers. This scoring system de...
[TRUNCATED DIAGNOSTIC VIEW]
